# 7) EE-3.2 Transmission line models — Exercises

**Goals:** line parameters and the π model, charging power of lines and cables, the Ferranti effect checked against pandapower, characteristic impedance and SIL, sizing a shunt reactor, and an exact equivalent π for long lines.

Voltages are line-to-line (kV), powers three-phase (MW, Mvar), as in EE-3.1. The helper `pandapower_line` builds a source–line–bus network with an optional load and shunt reactor.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
from scipy.optimize import brentq, fsolve

warnings.filterwarnings("ignore", category=FutureWarning)
OMEGA = 2 * math.pi * 50.0
R, X, C = 0.03, 0.30, 12e-9            # 400 kV overhead line per km: ohm, ohm, F (from the lecture)


def pandapower_line(length_km, r=R, x=X, c_nf=C * 1e9, g_us=0.0, load_mva=0j, shunt_mvar=0.0, vn_kv=400.0):
    """Source -> line -> receiving bus (optional load and shunt reactor) in pandapower; returns the net."""
    net = pp.create_empty_network()
    b1, b2 = pp.create_bus(net, vn_kv=vn_kv), pp.create_bus(net, vn_kv=vn_kv)
    pp.create_ext_grid(net, b1, vm_pu=1.0)
    pp.create_line_from_parameters(net, b1, b2, length_km=length_km, r_ohm_per_km=r, x_ohm_per_km=x,
                                   c_nf_per_km=c_nf, g_us_per_km=g_us, max_i_ka=5.0)
    if load_mva:
        pp.create_load(net, b2, p_mw=load_mva.real, q_mvar=load_mva.imag)
    if shunt_mvar:
        pp.create_shunt(net, b2, q_mvar=shunt_mvar)          # positive q_mvar = a reactor (absorbs Q)
    pp.runpp(net, numba=False)
    return net

### Warm-ups

**1. Line parameters**

Write `line_parameters(r, x, c, length_km)` returning the series impedance Z (Ω) and total shunt admittance Y (S) at 50 Hz (c in F/km).

In [ ]:
def line_parameters(r: float, x: float, c: float, length_km: float) -> tuple[complex, complex]:
    """Series impedance (ohm) and shunt admittance (S) of a line."""
    ...  # YOUR CODE HERE

In [ ]:
Z, Y = line_parameters(R, X, C, 300)
assert np.isclose(Z, 9 + 90j) and np.isclose(Y, 1j * OMEGA * 12e-9 * 300)
assert np.isclose(line_parameters(R, X, C, 600)[1], 2 * Y), "admittance grows with length"

**2. Charging power**

Write `charging_Mvar(V_kV, c, length_km)` ≈ V²·ωc·ℓ. Check the lecture's 300 km line, and answer the lecture's question about a 30 km, 220 kV export cable with c = 200 nF/km.

In [ ]:
def charging_Mvar(V_kV: float, c: float, length_km: float) -> float:
    """Approximate reactive power (Mvar) produced by a line's capacitance at voltage V_kV."""
    ...  # YOUR CODE HERE

In [ ]:
assert math.isclose(charging_Mvar(400, C, 300), 180.96, rel_tol=1e-3)
cable = charging_Mvar(220, 200e-9, 30)
assert cable > charging_Mvar(220, C, 30) * 15, "a cable charges more than 15 times an overhead line"
print(f"30 km, 220 kV cable: {cable:.1f} Mvar | the same length of overhead line: {charging_Mvar(220, C, 30):.1f} Mvar")

### Core

**3. The Ferranti rise from the two-port**

Write `abcd_pi(Z, Y)` and `no_load_voltage_pu(length_km)` = |V_r / V_s| with nothing connected at the far end. Check AD − BC = 1 and compare with pandapower.

In [ ]:
def abcd_pi(Z: complex, Y: complex) -> np.ndarray:
    """ABCD matrix of a nominal pi section."""
    ...  # YOUR CODE HERE


def no_load_voltage_pu(length_km: float) -> float:
    """Receiving-end voltage (pu of the sending voltage) of the unloaded 400 kV line."""
    ...  # YOUR CODE HERE

In [ ]:
assert np.isclose(np.linalg.det(abcd_pi(*line_parameters(R, X, C, 300))), 1.0)
for length in [150, 450]:
    ref = pandapower_line(length).res_bus.vm_pu[1]
    assert math.isclose(no_load_voltage_pu(length), ref, rel_tol=1e-9), f"{length} km must match pandapower"
assert no_load_voltage_pu(450) > no_load_voltage_pu(150) > 1.0, "the rise grows with length"
print(f"no-load rise: 150 km {no_load_voltage_pu(150):.4f} pu, 450 km {no_load_voltage_pu(450):.4f} pu")

**4. Characteristic impedance and SIL**

Write `surge_impedance(r, x, c)` (Ω, complex) and `sil_MW(V_kV, r, x, c)`. Compare the 400 kV overhead line with a 110 kV cable (r = 0.06, x = 0.12 Ω/km, c = 200 nF/km).

In [ ]:
def surge_impedance(r: float, x: float, c: float) -> complex:
    """Characteristic impedance Z_c = sqrt(z / y) (ohm)."""
    ...  # YOUR CODE HERE


def sil_MW(V_kV: float, r: float, x: float, c: float) -> float:
    """Surge impedance loading (MW)."""
    ...  # YOUR CODE HERE

In [ ]:
assert math.isclose(sil_MW(400, R, X, C), 565.78, rel_tol=1e-3), "the lecture's SIL"
zc_cable = abs(surge_impedance(0.06, 0.12, 200e-9))
assert zc_cable < abs(surge_impedance(R, X, C)) / 5, "cables have a much lower characteristic impedance"
print(f"overhead 400 kV: |Zc| = {abs(surge_impedance(R, X, C)):.0f} ohm, SIL {sil_MW(400, R, X, C):.0f} MW | "
      f"cable 110 kV: |Zc| = {zc_cable:.0f} ohm, SIL {sil_MW(110, 0.06, 0.12, 200e-9):.0f} MW")

**5. Size a shunt reactor**

The unloaded 300 km line rises to 1.054 pu. A shunt reactor at the receiving end absorbs reactive power; with susceptance b (S) it adds an admittance −jb there. Write `reactor_for_1pu(length_km)`: find b so that |V_r| = 1.0 pu (use the two-port: V_s = A·V_r + B·I_r with I_r = −jb·V_r, and `brentq`), and return the reactor's rating in Mvar at 400 kV (V²·b). Check it in pandapower with `shunt_mvar`.

In [ ]:
def reactor_for_1pu(length_km: float) -> float:
    """Shunt reactor rating (Mvar at 400 kV) that holds the unloaded line's far end at 1.0 pu."""
    ...  # YOUR CODE HERE

In [ ]:
q = reactor_for_1pu(300)
net = pandapower_line(300, shunt_mvar=q)
assert math.isclose(net.res_bus.vm_pu[1], 1.0, abs_tol=1e-6), "the reactor brings the far end back to 1.0 pu"
assert 0 < q < charging_Mvar(400, C, 300), "it needs less than the line's whole charging power"
print(f"300 km: a {q:.0f} Mvar reactor holds 1.0 pu (line charging {charging_Mvar(400, C, 300):.0f} Mvar)")

**6. Interpretation**

Using tasks 2, 4 and 5, explain why offshore wind farms more than a few tens of kilometres from shore increasingly use HVDC export cables, and why AC cable connections that are used need shunt reactors at both ends.

*Your answer here.*

### Challenge

**7. An exact equivalent π for long lines**

pandapower uses one nominal π per line, which overstates the Ferranti rise of a 600 km line by 0.8 % (lecture). Fix that without changing the tool: build the **equivalent π** whose two-port equals the exact distributed line, Z' = B_exact and Y'/2 = (A_exact − 1) / B_exact, and enter it into pandapower as a 1 km "line" (r, x from Z'; c and g from Y' — `g_us_per_km` takes the real part in µS). Then check:

1. unloaded: pandapower's far-end voltage must equal the exact model's 1/|A|;
2. loaded with 300 MW + 50 Mvar: pandapower must match the exact two-port solved directly with `fsolve` (V_s = A·V_r + B·I_r with √3·I_r = (S/V_r)\* in kV, MVA, kA — EE-3.1).

In [ ]:
def abcd_exact(length_km: float) -> np.ndarray:
    """ABCD matrix of the distributed-parameter 400 kV line."""
    ...  # YOUR CODE HERE


def equivalent_pi_params(length_km: float) -> dict:
    """pandapower parameters (per km, for a 1 km line) of the exact equivalent pi."""
    ...  # YOUR CODE HERE


def exact_receiving_voltage_kV(length_km: float, load_mva: complex, V_s_kV: float = 400.0) -> complex:
    """Receiving-end voltage phasor (kV) of the exact line for a given load, by solving the two-port."""
    ...  # YOUR CODE HERE

In [ ]:
eq = equivalent_pi_params(600)
unloaded = pandapower_line(1, r=eq["r"], x=eq["x"], c_nf=eq["c_nf"], g_us=eq["g_us"])
assert math.isclose(unloaded.res_bus.vm_pu[1], abs(1 / abcd_exact(600)[0, 0]), rel_tol=1e-8), "unloaded: exact"

S = 300 + 50j
loaded = pandapower_line(1, r=eq["r"], x=eq["x"], c_nf=eq["c_nf"], g_us=eq["g_us"], load_mva=S)
V_exact = exact_receiving_voltage_kV(600, S)
assert math.isclose(loaded.res_bus.vm_pu[1], abs(V_exact) / 400, rel_tol=1e-7), "loaded: magnitude matches"
assert math.isclose(loaded.res_bus.va_degree[1], np.angle(V_exact, deg=True), abs_tol=1e-5), "loaded: angle matches"

nominal = pandapower_line(600, load_mva=S).res_bus.vm_pu[1]
print(f"600 km, 300 MW + 50 Mvar: exact {abs(V_exact) / 400:.5f} pu | equivalent pi in pandapower "
      f"{loaded.res_bus.vm_pu[1]:.5f} pu | one nominal pi {nominal:.5f} pu")

**8. Interpretation**

When is the single nominal π in pandapower or PyPSA good enough, and what two remedies do you now have when it is not? Which would you choose for a 600 km line in a large network model that others must maintain?

*Your answer here.*